In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# Config
# ============================================================
BASE_DIR = Path("./")

INPUT_SPILLOVER = BASE_DIR / "gfevd_all.npy"   # shape: (T, k, k)
INPUT_DATE = BASE_DIR / "tvpvar_input_scaled.csv"

OUT_DIR = BASE_DIR / "event_detection_outputs"
OUT_DIR.mkdir(exist_ok=True)

WINDOW_LIST = [0, 3, 5]
THRESHOLD_Q = 0.95

# ============================================================
# 1. Load
# ============================================================
gfevd = np.load(INPUT_SPILLOVER)   # (T, k, k)

df_date = pd.read_csv(INPUT_DATE)
df_date["Date"] = pd.to_datetime(df_date["Date"])
dates = df_date["Date"].values[-gfevd.shape[0]:]  # align

T, k, _ = gfevd.shape

print(f"[INFO] T={T}, k={k}")

# ============================================================
# 2. Δ spillover 계산
# ============================================================
delta = np.abs(gfevd[1:] - gfevd[:-1])   # (T-1, k, k)

# 자기 자신 제외
mask = ~np.eye(k, dtype=bool)

delta_total = delta[:, mask].reshape(T-1, -1).sum(axis=1)

dates_delta = dates[1:]

df_base = pd.DataFrame({
    "Date": dates_delta,
    "delta_total": delta_total
})

# 저장
df_base.to_csv(OUT_DIR / "delta_total_raw.csv", index=False)

print("[INFO] delta_total computed")

# ============================================================
# 3. Event Detection 함수
# ============================================================
def detect_events(series, dates, window, q):
    df = pd.DataFrame({
        "Date": dates,
        "value": series
    }).copy()

    # smoothing
    if window > 0:
        df["value_smooth"] = df["value"].rolling(window=window, center=True).mean()
    else:
        df["value_smooth"] = df["value"]

    # threshold
    threshold = df["value_smooth"].quantile(q)

    df["event"] = df["value_smooth"] > threshold

    # event 날짜
    event_dates = df.loc[df["event"], "Date"]

    return df, event_dates, threshold

# ============================================================
# 4. window별 실행
# ============================================================
summary = []

for w in WINDOW_LIST:
    print(f"\n[INFO] Running window={w}")

    df_res, event_dates, th = detect_events(
        delta_total,
        dates_delta,
        window=w,
        q=THRESHOLD_Q
    )

    # 저장
    df_res.to_csv(OUT_DIR / f"delta_total_window{w}.csv", index=False)

    event_df = pd.DataFrame({"Date": event_dates})
    event_df.to_csv(OUT_DIR / f"events_window{w}.csv", index=False)

    print(f"[INFO] window={w} threshold={th:.4f}, events={len(event_dates)}")

    summary.append({
        "window": w,
        "threshold": th,
        "num_events": len(event_dates)
    })

# ============================================================
# 5. summary 저장
# ============================================================
df_summary = pd.DataFrame(summary)
df_summary.to_csv(OUT_DIR / "event_summary.csv", index=False)

print("\n[INFO] Done")

[INFO] T=1035, k=7
[INFO] delta_total computed

[INFO] Running window=0
[INFO] window=0 threshold=1.4256, events=52

[INFO] Running window=3
[INFO] window=3 threshold=1.1423, events=52

[INFO] Running window=5
[INFO] window=5 threshold=1.0377, events=52

[INFO] Done
